# 5장 · 파일·표·텍스트를 읽고 정제 결과 저장하기

기록표를 받았는데 id가 2인 첫 행은 시간이 비어 있고 그다음 행에는 20분이 적혀 있습니다. 중복부터 지우면 쓸 수 있는 20분이 사라집니다. 어느 행을 먼저 남길지 작은 표로 확인합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 pandas 2.2.3입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## CSV에서 글자와 빈칸 읽기

이 CSV에서는 쉼표가 한 줄 안의 칸을 나누고 줄바꿈이 행을 나눕니다. 노트북에 넣은 여섯 줄은 예제 records.csv와 같습니다. StringIO로 파일처럼 열어 보므로 경로를 설정할 필요가 없습니다.



**볼 곳**　‘CSV 행’ 안에서 ‘빈 칸’를 거쳐 어느 값이 오른쪽에 남는지 보세요.

**애니메이션의 의미**　CSV의 빈 칸은 먼저 빈 문자열로 읽힙니다.


### 셀 01 · 여섯 줄의 기록

둘째 데이터 행에서 minutes를 꺼내 보세요. 빈 문자열은 화면에 잘 안 보이므로 repr로 표시합니다.

In [1]:
import csv
from io import StringIO
csv_text = """id,title,minutes
1,  Python   기초  ,30
2,Git 연습,
2,Git 연습,20
3,,15
4,CSV  읽기,40
4,CSV  읽기,40
"""
rows = list(csv.DictReader(StringIO(csv_text)))
print(repr(rows[1]["minutes"]))

''


작은따옴표 두 개는 길이 0인 문자열입니다. 0분이라는 뜻으로 고치면 누락된 값과 실제 0분을 구분할 수 없습니다.

### 셀 02 · 문장이 적힌 열 정리하기

첫 행 제목 사이에 공백이 몇 칸 있는지 눈으로 세지 않고 단어를 다시 잇습니다.

In [2]:
print(" ".join(rows[0]["title"].split()))

Python 기초


Python 기초가 됩니다. id는 행을 다시 찾을 때 쓸 이름이므로 제목 옆에 남겨 둡니다.

## 유효한 행을 남긴 뒤 중복 제거하기

시간이 비었거나 제목이 비면 그 행은 보류합니다. 이 표의 시간은 0~480 사이의 정수만 허용합니다. 남은 행 안에 같은 id가 다시 나타나면 먼저 본 유효 행을 씁니다.



**볼 곳**　‘원본 행’가 가운데 ‘유효성 검사’에 도착한 뒤 오른쪽에 어떤 결과가 남는지 보세요.

**애니메이션의 의미**　유효하지 않은 행을 뺀 뒤 중복을 찾습니다.


### 셀 03 · 누락을 걸러낸 기록

아래 함수는 이 작은 CSV의 정수 표기만 읽습니다. 숫자 변환과 범위 검사까지 성공한 행을 모으세요.

CSV의 minutes는 문자열입니다. clean_row는 '30'을 30으로 바꾸지만 '30.0'에서는 int 변환이 실패해 행을 제외합니다. 뒤의 판다스 예시는 먼저 숫자로 읽어 정수와 값이 같은 30.0을 허용합니다. 팀 자료에는 어느 표기까지 허용할지 한 가지 규칙을 정하세요.

In [3]:
def clean_row(row):
    title = " ".join(row["title"].split())
    try:
        minutes = int(row["minutes"])
    except ValueError:
        return None
    if not row["id"].strip() or not title or not 0 <= minutes <= 480:
        return None
    return {"id": row["id"].strip(), "title": title, "minutes": minutes}

valid = []
for row in rows:
    cleaned = clean_row(row)
    if cleaned is not None:
        valid.append(cleaned)
print([row["id"] for row in valid])

['1', '2', '4', '4']


1, 2, 4, 4가 남습니다. 먼저 나타난 2번 행은 시간이 없어서 빠졌고 3번은 제목이 비었습니다. 4번 두 건은 내용이 유효하므로 아직 둘 다 있습니다.

### 과제 04 · 첫 유효 행만 남기기

사전 by_id에 아직 없는 id만 저장하고 남은 기록의 시간을 출력하세요.

In [4]:
by_id = {}
# valid에서 첫 유효 행을 id별로 모으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>by_id = {}
for row in valid:
    if row[&quot;id&quot;] not in by_id:
        by_id[row[&quot;id&quot;]] = row
print([row[&quot;minutes&quot;] for row in by_id.values()])</code></pre>
<p>실행 결과</p><pre><code>[30, 20, 40]</code></pre>
<p>30, 20, 40이 남습니다. 입력 여섯 행 가운데 무효 두 행과 중복 한 행을 빼면 세 행입니다.</p>
</details>

## 판다스 표에서도 같은 순서로 읽기

표 전체를 다룰 때는 Pandas의 DataFrame을 씁니다. 수업용 커널에 pandas가 있으면 아래 셀이 실제 열을 선택합니다. 설치되지 않은 환경에서도 앞의 CSV 실습은 그대로 동작합니다.



**볼 곳**　‘표의 각 행’ 안에서 ‘조건 마스크’를 거쳐 어느 값이 오른쪽에 남는지 보세요.

**애니메이션의 의미**　행마다 참·거짓 하나가 대응합니다.


### 셀 05 · 시간 열에서 빈칸 찾기

문자열로 읽은 뒤 숫자로 바꿀 수 없는 값을 결측값으로 표시합니다.

In [5]:
from importlib.util import find_spec
if find_spec("pandas") is not None:
    import pandas as pd
    frame = pd.read_csv(StringIO(csv_text), dtype=str, keep_default_na=False)
    minutes = pd.to_numeric(frame["minutes"], errors="coerce")
    print(minutes.isna().tolist())
else:
    print("이 커널에는 pandas가 없습니다. CSV의 두 번째 데이터 행에서 시간을 확인하세요.")

[False, True, False, False, False, False]


Pandas가 있으면 여섯 값 중 둘째만 True입니다. 이는 시간 칸만 검사한 결과라서 제목이 빈 행은 아직 골라내지 않았습니다. 전체 예제의 clean_table은 제목·범위·정수 조건을 합친 뒤 중복을 제거합니다.

## 저장한 기록을 다시 읽기

메모리 안의 사전은 커널을 닫으면 사라집니다. 임시 폴더에 실제 파일을 저장하고 같은 인코딩으로 다시 열어 첫 유효 행의 값이 남았는지 확인합니다.



**볼 곳**　‘정제한 기록’를 ‘파일 저장’에 남기는 시점과 ‘다시 읽은 기록’를 확인하는 시점을 구분하세요.

**애니메이션의 의미**　다시 읽은 제목과 시간이 저장 전 값과 맞아야 합니다.


### 셀 06 · UTF-8 JSON 한 건 되읽기

파일을 여는 기준 폴더와 이름을 Path로 합칩니다. 이번 입력은 valid의 첫 행이므로 앞 과제를 풀기 전에도 실행할 수 있습니다.

In [6]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as folder:
    target = Path(folder) / "record.json"
    target.write_text(json.dumps(valid[0], ensure_ascii=False), encoding="utf-8")
    restored = json.loads(target.read_text(encoding="utf-8"))
    print(restored["title"], restored["minutes"])

Python 기초 30


‘Python 기초 30’이 나옵니다. 글자를 쓸 때와 읽을 때 같은 UTF-8을 지정했고, loads를 거쳐 정수 시간도 되찾았습니다. 임시 폴더는 셀이 끝나면 정리됩니다.

## 돌아보기

처음 받은 CSV에서 특정 행을 지웠다면 이유를 말할 수 있어야 합니다. 2번 첫 행과 4번 마지막 행은 각각 다른 단계에서 빠집니다. 처리 결과를 JSON으로 저장할 때도 id와 정수 시간을 함께 남기면 원자료까지 돌아갈 수 있습니다.

### 6행이 3행으로 정리되는 과정



**볼 곳**　유효성 검사와 중복 제거를 지날 때 남은 행 수와 ID를 읽으세요.

**애니메이션의 의미**　잘못된 두 행을 먼저 빼고 유효한 중복 한 행을 제거하면 처음 여섯 행 중 세 행이 남습니다.
